In [4]:
! pip install statsmodels
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error

def lstm_har_stacked_ensemble(X, y, features, lstm_lags=10, test_size=0.2):
    """
    HAR + LSTM ensemble using meta-model stacking.

    Returns:
    - final_pred: meta-model predictions
    - y_test: true values
    - meta_model: trained linear regressor
    """

    if isinstance(y, pd.Series):
        y = y.values
    if not isinstance(X, pd.DataFrame):
        raise ValueError("X must be a DataFrame.")

    n = len(y)
    split_idx = int(n * (1 - test_size))

    # HAR Model
    X_har = sm.add_constant(X[features])
    har_model = sm.OLS(y[:split_idx], X_har.iloc[:split_idx]).fit()
    har_pred = har_model.predict(X_har.iloc[split_idx:])

    # LSTM Prep
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X[features])

    X_seq, y_seq = [], []
    for i in range(lstm_lags, len(X_scaled)):
        X_seq.append(X_scaled[i - lstm_lags:i])
        y_seq.append(y[i])
    X_seq, y_seq = np.array(X_seq), np.array(y_seq)

    split_lstm = int(len(y_seq) * (1 - test_size))
    X_train_lstm, X_test_lstm = X_seq[:split_lstm], X_seq[split_lstm:]
    y_train_lstm, y_test_lstm = y_seq[:split_lstm], y_seq[split_lstm:]

    # LSTM Model
    model = Sequential()
    model.add(LSTM(50, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train_lstm, y_train_lstm, epochs=20, batch_size=16, verbose=0)
    lstm_pred = model.predict(X_test_lstm).flatten()

    # Align HAR predictions
    har_pred = har_pred[-len(lstm_pred):]
    y_test = y_test_lstm

    # Meta-model (stacking) using Linear Regression
    stack_input = np.vstack([har_pred, lstm_pred]).T  # shape: (n_samples, 2)
    meta_model = LinearRegression().fit(stack_input, y_test)
    final_pred = meta_model.predict(stack_input)

    return final_pred, y_test, meta_model


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.9/232.9 kB 2.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
2025-06-30 13:47:09.726858: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-30 13:47:10.116843: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-30 13:47:10.117456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-30 13:47:10.118823: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory f

In [ ]:
import numpy as np
import pandas as pd
# Load data
y = pd.read_csv("/work/y_combined_scaled_new.csv", index_col=0)
X = pd.read_csv("/work/X_combined_scaled_new.csv", index_col=0)

X = X.drop("DS", axis=1)
X = X.drop("DPR", axis=1)

In [ ]:
features = X.columns
lstm_har_stacked_ensemble(X, y, features, lstm_lags=10, test_size=0.2):